# RAILGUN+ — Notebook 1: Setup, Generate Data, Train

Runs on **Google Colab** (GPU runtime). Edit code in VS Code -> push -> re-clone here. Never edit code in Colab.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/railgun-plus'
DATA_DIR = f'{DRIVE_ROOT}/data/shards'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Data ->', DATA_DIR)

## 2. Clone your repo
Replace with YOUR GitHub URL. Re-run after every push from VS Code.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/railgun-plus.git'  # <-- EDIT
%cd /content
![ -d railgun-plus ] && (cd railgun-plus && git pull) || git clone $REPO_URL
%cd /content/railgun-plus

## 3. Install dependencies

In [ ]:
!pip install -q pogema pogema-toolbox pyyaml tqdm
import sys
sys.path.insert(0, '/content/railgun-plus/src')
print('ok')

## 4. Self-test

In [ ]:
!cd /content/railgun-plus && PYTHONPATH=src python -m pytest tests/ -q

## 5. Generate PIBT expert data (Route C)
Start small (50), verify, then scale up.

In [ ]:
!cd /content/railgun-plus && PYTHONPATH=src python scripts/generate_data.py \
    --out "$DATA_DIR" --instances 50 --map-size 32 --density 0.2 --agents 32 --seed 0

In [ ]:
# Scale up later across agent counts:
# for agents in [16,32,64,96]:
#     !cd /content/railgun-plus && PYTHONPATH=src python scripts/generate_data.py \
#         --out "$DATA_DIR" --instances 200 --map-size 32 --density 0.2 --agents {agents} --seed {agents}

## 6. Train

In [ ]:
import torch
from railgun_plus.models import RailgunUNet
from railgun_plus.data.dataset import ShardedMapfDataset
from railgun_plus.train import train

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset = ShardedMapfDataset(DATA_DIR)
print('samples:', len(dataset))
model = RailgunUNet(in_channels=6, num_actions=5, base=64)
print('params:', f'{model.count_params()/1e6:.1f}M')
history = train(model, dataset, epochs=10, batch_size=8, lr=1e-3,
                weight_decay=1e-3, device=device, ckpt_dir=CKPT_DIR)

## 7. Plot loss

In [ ]:
import matplotlib.pyplot as plt
l=[h['train_loss'] for h in history]
plt.plot(range(1,len(l)+1),l,marker='o');plt.xlabel('epoch');plt.ylabel('loss');plt.grid(True);plt.show()